## Code to Convert CelebA Dataset to Standard format of YOLOv8 Model

In [ ]:
import os
import cv2
import pandas as pd
import shutil

image_dir = "path_to/CelebA_dataset/img_align_celeba"
csv_file = "path_to/CelebA_dataset/list_landmarks_align_celeba.csv"

output_dataset = "annotated_dataset"

# Output folders
train_img_dir = os.path.join(output_dataset, "images/train")
train_label_dir = os.path.join(output_dataset, "labels/train")
val_img_dir = os.path.join(output_dataset, "images/val")
val_label_dir = os.path.join(output_dataset, "labels/val")

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_label_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# Read CSV
df = pd.read_csv(csv_file)

# Shuffle rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split ratio
val_ratio = 0.2
val_count = int(len(df) * val_ratio)

for idx, row in df.iterrows():
    img_name = row["image_id"]
    img_path = os.path.join(image_dir, img_name)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Get x/y coordinates
    xs = [
        row["lefteye_x"], row["righteye_x"],
        row["nose_x"],
        row["leftmouth_x"], row["rightmouth_x"]
    ]
    ys = [
        row["lefteye_y"], row["righteye_y"],
        row["nose_y"],
        row["leftmouth_y"], row["rightmouth_y"]
    ]

    xmin = min(xs)
    xmax = max(xs)
    ymin = min(ys)
    ymax = max(ys)

    bw = xmax - xmin
    bh = ymax - ymin

    # Optional: add padding for better face detection
    pad_w = bw * 0.2
    pad_h = bh * 0.2

    xmin = max(0, xmin - pad_w)
    ymin = max(0, ymin - pad_h)
    xmax = min(w, xmax + pad_w)
    ymax = min(h, ymax + pad_h)

    bw = xmax - xmin
    bh = ymax - ymin

    xc = (xmin + bw/2) / w
    yc = (ymin + bh/2) / h
    bw /= w
    bh /= h

    label_text = f"0 {xc} {yc} {bw} {bh}"

    # Decide train or val
    if idx < val_count:
        img_out = os.path.join(val_img_dir, img_name)
        label_out = os.path.join(val_label_dir, img_name.replace(".jpg",".txt"))
    else:
        img_out = os.path.join(train_img_dir, img_name)
        label_out = os.path.join(train_label_dir, img_name.replace(".jpg",".txt"))

    shutil.copy(img_path, img_out)
    with open(label_out, "w") as f:
        f.write(label_text)

print("CelebA conversion + train/val split complete")